# Regresja logistyczna jako pipeline
Ten sam model co w przykladowa regresja logistyczna, ale ladnie ulozony w pipeline

In [47]:
#importuje napisane przez nas funkcje i je aktualizuje na bieżąco, dzięki czemu nie trzeba restartować kernela przy każdej zmianie w funkcjach
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from funkcje.funkcje import *
from funkcje.wskazniki_kredytowe import oblicz_wskazniki

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV

## Transformer przygotowania danych



In [49]:
class LogisticRegressionUjemne(LogisticRegression):
    def fit(self, X, y, sample_weight=None):
        super().fit(X, y, sample_weight=sample_weight)
        self.coef_[self.coef_ > 0] = 0
        return self


In [25]:

class PrzygotowanieCech(BaseEstimator, TransformerMixin):
    def __init__(
        self,
        kolumny_do_pominiecia=None,
        prog_korelacji=0.9,
        prog_auc=0.6,
        tylko_wskazniki=False,
        prog_nan=0.5,
        prog_zer=0.5,
        prog_monotonicznosci=0.1,
        liczba_kwantyli=10,
    ):
        self.kolumny_do_pominiecia = kolumny_do_pominiecia
        self.prog_korelacji = prog_korelacji
        self.prog_auc = prog_auc
        self.tylko_wskazniki = tylko_wskazniki
        self.prog_nan = prog_nan
        self.prog_zer = prog_zer
        self.prog_monotonicznosci = prog_monotonicznosci
        self.liczba_kwantyli = liczba_kwantyli

    def _przygotuj_baze(self, X):
        if not isinstance(X, pd.DataFrame):
            raise TypeError("Dane wejściowe muszą być obiektem pandas.DataFrame.")

        pomin = self.kolumny_do_pominiecia or []
        brakujace = sorted(set(pomin) - set(X.columns))
        if brakujace:
            raise ValueError(f"Brak wymaganych kolumn do pominięcia: {brakujace}")

        baza = X.drop(columns=pomin).copy()
        kolumny_poczatkowe = set(baza.columns)

        baza = oblicz_wskazniki(baza)

        if self.tylko_wskazniki:
            nowe_kolumny = [
                kolumna for kolumna in baza.columns
                if kolumna not in kolumny_poczatkowe
            ]
            baza = baza[nowe_kolumny]

        return baza

    @staticmethod
    def _kolumny_numeryczne(X):
        kolumny = X.select_dtypes(include=["float"]).columns.tolist()
        if "wsk_liczba_dni_istnienia" in X.columns and "wsk_liczba_dni_istnienia" not in kolumny:
            kolumny.append("wsk_liczba_dni_istnienia")
        return kolumny

    def _odrzuc_kolumny_niskiego_auc(self, X, y):
        rows = []
        for kolumna in X.columns:
            maska = X[kolumna].notna()
            if maska.sum() == 0 or y.loc[maska].nunique() < 2:
                continue
            auc = roc_auc_score(y.loc[maska], X.loc[maska, kolumna])
            rows.append((kolumna, max(auc, 1 - auc), auc >= 0.5))

        if not rows:
            raise ValueError("Nie udało się policzyć AUC dla żadnej cechy.")

        tabela = pd.DataFrame(rows, columns=["cecha", "auc", "zgodny_kierunek"])
        do_usuniecia = tabela.loc[tabela["auc"] < self.prog_auc, "cecha"].tolist()
        do_odwrocenia = tabela.loc[
            (tabela["auc"] >= self.prog_auc) & tabela["zgodny_kierunek"], "cecha"
        ].tolist()
        return do_usuniecia, do_odwrocenia

    def fit(self, X, y):
        y = pd.Series(y, index=X.index if isinstance(X, pd.DataFrame) else None)
        baza = self._przygotuj_baze(X)
        self.kolumny_numeryczne_ = self._kolumny_numeryczne(baza)
        if not self.kolumny_numeryczne_:
            raise ValueError("Nie znaleziono cech numerycznych.")

        numeryczne = baza[self.kolumny_numeryczne_].copy()
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)

        maska_nan = numeryczne.isna().mean() <= self.prog_nan
        maska_zer = numeryczne.eq(0).mean() <= self.prog_zer
        maska_jakosci = maska_nan & maska_zer

        self.kolumny_zbyt_wielu_brakow_ = numeryczne.columns[~maska_jakosci].tolist()
        numeryczne = numeryczne.loc[:, maska_jakosci]

        po_korelacji = usuwanie_najbardziej_scorrelowanych(
            numeryczne, prog=self.prog_korelacji
        )
        self.kolumny_po_korelacji_ = po_korelacji.columns.tolist()
        nie_monotoniczne = wykryj_niemonotoniczne(po_korelacji, y, prog=self.prog_monotonicznosci, liczba_kwantyli=self.liczba_kwantyli)
        self.kolumny_niemonotoniczne_ = nie_monotoniczne

        if self.kolumny_niemonotoniczne_:
            po_zawiasach=dodaj_cechy_zawiasowe(po_korelacji, nie_monotoniczne)
        else:
            po_zawiasach= po_korelacji

        self.kolumny_do_usuniecia_, self.kolumny_do_odwrocenia_ = (
            self._odrzuc_kolumny_niskiego_auc(po_zawiasach, y)
        )
        wynik = po_zawiasach.drop(columns=self.kolumny_do_usuniecia_)
        

        skosnosc = wynik.skew()
        self.kolumny_do_log_ = [
            kolumna for kolumna in wynik.columns
            if skosnosc[kolumna] > 2 and wynik[kolumna].min() >= 0
        ]
        wynik[self.kolumny_do_log_] = np.log1p(wynik[self.kolumny_do_log_])
        wynik[self.kolumny_do_odwrocenia_] = -wynik[self.kolumny_do_odwrocenia_]

        wynik = wynik.rename(columns={
            kolumna: f"-{kolumna}"
            for kolumna in self.kolumny_do_odwrocenia_
        })


        self.kolumny_wynikowe_ = wynik.columns.tolist()
        return self

    def transform(self, X):
        baza = self._przygotuj_baze(X)
        numeryczne = baza.reindex(columns=self.kolumny_numeryczne_)
        numeryczne = numeryczne.replace([np.inf, -np.inf], np.nan)
        numeryczne = numeryczne.drop(columns=self.kolumny_zbyt_wielu_brakow_)
        numeryczne = numeryczne.reindex(columns=self.kolumny_po_korelacji_)

        if self.kolumny_niemonotoniczne_:
            numeryczne = dodaj_cechy_zawiasowe(
                numeryczne, self.kolumny_niemonotoniczne_
            )

        numeryczne = numeryczne.drop(columns=self.kolumny_do_usuniecia_)
        
        numeryczne[self.kolumny_do_log_]=numeryczne[self.kolumny_do_log_].clip(lower=0)
        numeryczne[self.kolumny_do_log_] = np.log1p(numeryczne[self.kolumny_do_log_])
        numeryczne[self.kolumny_do_odwrocenia_] = -numeryczne[self.kolumny_do_odwrocenia_]

        numeryczne = numeryczne.rename(columns={
            kolumna: f"-{kolumna}"
            for kolumna in self.kolumny_do_odwrocenia_
        })

        return numeryczne.reindex(columns=self.kolumny_wynikowe_)


## Trenowanie i użycie pipeline'u

In [50]:
df = pd.read_csv("../zbior_1.csv")
TARGET_COLUMN = df.columns[-1]
KOLUMNY_DO_POMINIECIA = [
    "szczegolnaFormaPrawna_Symbol",
    "schemat_wsk_rzis",
    "schemat_wsk_bilans",
]

X = df.drop(columns=TARGET_COLUMN)
y = df[TARGET_COLUMN]
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=2137, stratify=y
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=2137, stratify=y_temp
)

pipeline = Pipeline([
    ("przygotowanie_cech", PrzygotowanieCech(KOLUMNY_DO_POMINIECIA)),
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegressionUjemne(
        l1_ratio=1, solver="liblinear", C=100, random_state=2137, max_iter=1000
    )),
])

pipeline.fit(X_train, y_train)
prawdopodobienstwo = pipeline.predict_proba(X_valid)[:, 1]
predykcja = pipeline.predict(X_valid)

print(f"AUC: {roc_auc_score(y_valid, prawdopodobienstwo):.3f}")

AUC: 0.809


In [51]:
#optymalizacja parametrów pipeline'u, ale dodatnie be
param_grid = {
    "przygotowanie_cech__prog_auc": [0.5, 0.55, 0.6],
    "przygotowanie_cech__prog_korelacji": [0.7,0.8 ,0.9],
    "przygotowanie_cech__tylko_wskazniki": [False, True],
    "przygotowanie_cech__liczba_kwantyli": [5, 10],
    "przygotowanie_cech__prog_monotonicznosci": [0.1, 0.2, 0.3],
    "przygotowanie_cech__prog_nan": [0.35, 0.5,0.65],
    "przygotowanie_cech__prog_zer": [0.3,0.4, 0.5],
    "classifier__C": [0.1,1,2, 10],
}
wyniki = []
najlepszy_auc = -float("inf")
najlepszy_model = None
najlepsze_parametry = None

kombinacje = list(ParameterGrid(param_grid))

for i, parametry in enumerate(kombinacje, start=1):
    model = clone(pipeline)
    model.set_params(**parametry)


    model.fit(X_train, y_train)

    prawdopodobienstwo = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, prawdopodobienstwo)
    betas = model.named_steps["classifier"].coef_[0]
    if np.sum(betas > 0) == 0:
        wyniki.append({**parametry, "auc": auc})

        if auc > najlepszy_auc:
            najlepszy_auc = auc
            najlepszy_model = model
            najlepsze_parametry = parametry

    print(f"[{i}/{len(kombinacje)}] AUC: {auc:.4f}")

wyniki_df = (
    pd.DataFrame(wyniki)
    .sort_values("auc", ascending=False)
    .reset_index(drop=True)
)

print("\nNajlepsze parametry:")
print(najlepsze_parametry)
print(f"\nNajlepszy ROC AUC: {najlepszy_auc:.4f}")

with pd.option_context("display.max_rows", 100):
    display(wyniki_df)

[1/3888] AUC: 0.7931
[2/3888] AUC: 0.7348
[3/3888] AUC: 0.8057
[4/3888] AUC: 0.7503
[5/3888] AUC: 0.8109
[6/3888] AUC: 0.7503
[7/3888] AUC: 0.7987
[8/3888] AUC: 0.7347
[9/3888] AUC: 0.8057
[10/3888] AUC: 0.7503
[11/3888] AUC: 0.8109
[12/3888] AUC: 0.7503
[13/3888] AUC: 0.8014
[14/3888] AUC: 0.7377
[15/3888] AUC: 0.8057
[16/3888] AUC: 0.7475
[17/3888] AUC: 0.8109
[18/3888] AUC: 0.7475
[19/3888] AUC: 0.8135
[20/3888] AUC: 0.7345
[21/3888] AUC: 0.8021
[22/3888] AUC: 0.7666
[23/3888] AUC: 0.8111
[24/3888] AUC: 0.7666
[25/3888] AUC: 0.8069
[26/3888] AUC: 0.7345
[27/3888] AUC: 0.8021
[28/3888] AUC: 0.7666
[29/3888] AUC: 0.8111
[30/3888] AUC: 0.7666
[31/3888] AUC: 0.8080
[32/3888] AUC: 0.7555
[33/3888] AUC: 0.8021
[34/3888] AUC: 0.7671
[35/3888] AUC: 0.8111
[36/3888] AUC: 0.7671
[37/3888] AUC: 0.8052
[38/3888] AUC: 0.7176
[39/3888] AUC: 0.8114
[40/3888] AUC: 0.7620
[41/3888] AUC: 0.8163
[42/3888] AUC: 0.7620
[43/3888] AUC: 0.8005
[44/3888] AUC: 0.7176
[45/3888] AUC: 0.8115
[46/3888] AUC: 0.76

,classifier__C,przygotowanie_cech__liczba_kwantyli,przygotowanie_cech__prog_auc,przygotowanie_cech__prog_korelacji,przygotowanie_cech__prog_monotonicznosci,przygotowanie_cech__prog_nan,przygotowanie_cech__prog_zer,przygotowanie_cech__tylko_wskazniki,auc
0,10.0,10,0.6,0.7,0.1,0.50,0.3,False,0.828659
1,2.0,10,0.6,0.7,0.1,0.50,0.3,False,0.828476
2,1.0,10,0.6,0.7,0.1,0.50,0.3,False,0.828415
3,0.1,10,0.6,0.7,0.1,0.50,0.3,False,0.823537
4,0.1,10,0.6,0.7,0.1,0.35,0.3,False,0.823232
...,...,...,...,...,...,...,...,...,...
3883,2.0,10,0.5,0.7,0.3,0.50,0.3,True,0.671890
3884,10.0,10,0.5,0.7,0.3,0.35,0.3,True,0.671037
3885,10.0,10,0.5,0.8,0.3,0.35,0.3,True,0.671037
3886,10.0,10,0.5,0.7,0.3,0.50,0.3,True,0.670061


In [52]:
with pd.option_context("display.max_rows", 100, "display.max_columns", None):
    display(wyniki_df.head(100))

,classifier__C,przygotowanie_cech__liczba_kwantyli,przygotowanie_cech__prog_auc,przygotowanie_cech__prog_korelacji,przygotowanie_cech__prog_monotonicznosci,przygotowanie_cech__prog_nan,przygotowanie_cech__prog_zer,przygotowanie_cech__tylko_wskazniki,auc
0,10.0,10,0.60,0.7,0.1,0.50,0.3,False,0.828659
1,2.0,10,0.60,0.7,0.1,0.50,0.3,False,0.828476
2,1.0,10,0.60,0.7,0.1,0.50,0.3,False,0.828415
3,0.1,10,0.60,0.7,0.1,0.50,0.3,False,0.823537
4,0.1,10,0.60,0.7,0.1,0.35,0.3,False,0.823232
5,0.1,10,0.60,0.7,0.1,0.65,0.3,False,0.823232
6,0.1,10,0.60,0.9,0.1,0.65,0.3,False,0.822805
7,1.0,10,0.60,0.7,0.1,0.35,0.3,False,0.822683
8,0.1,5,0.55,0.7,0.2,0.35,0.3,False,0.822561
9,1.0,5,0.55,0.8,0.2,0.35,0.3,False,0.821220


In [57]:
#unikalne cechy w kazdej kolumnie w 26 pierwszych wierszach wyników i wpisne ich do slownika
unikalne_cechy = {}
for col in wyniki_df.columns:
    if col == "auc":
        continue
    unique_values = wyniki_df[col].head(26).unique()
    print(f"Kolumna: {col}, Unikalne wartości (pierwsze 26 wierszy): {unique_values}")
    unikalne_cechy[col] = list(unique_values)


Kolumna: classifier__C, Unikalne wartości (pierwsze 26 wierszy): [10.   2.   1.   0.1]
Kolumna: przygotowanie_cech__liczba_kwantyli, Unikalne wartości (pierwsze 26 wierszy): [10  5]
Kolumna: przygotowanie_cech__prog_auc, Unikalne wartości (pierwsze 26 wierszy): [0.6  0.55]
Kolumna: przygotowanie_cech__prog_korelacji, Unikalne wartości (pierwsze 26 wierszy): [0.7 0.9 0.8]
Kolumna: przygotowanie_cech__prog_monotonicznosci, Unikalne wartości (pierwsze 26 wierszy): [0.1 0.2]
Kolumna: przygotowanie_cech__prog_nan, Unikalne wartości (pierwsze 26 wierszy): [0.5  0.35 0.65]
Kolumna: przygotowanie_cech__prog_zer, Unikalne wartości (pierwsze 26 wierszy): [0.3 0.5 0.4]
Kolumna: przygotowanie_cech__tylko_wskazniki, Unikalne wartości (pierwsze 26 wierszy): [False]


In [58]:
#zrobinie grid search z najlepszymi parametrami, zeby zminiejszyc przeuczenie 
param_grid = unikalne_cechy
model = GridSearchCV(estimator=pipeline, param_grid=param_grid, scoring="roc_auc", cv=5, verbose=10, n_jobs=-1)
model.fit(X_train, y_train)
model.best_params_, model.best_score_

Fitting 5 folds for each of 864 candidates, totalling 4320 fits


({'classifier__C': np.float64(0.1),
  'przygotowanie_cech__liczba_kwantyli': np.int64(10),
  'przygotowanie_cech__prog_auc': np.float64(0.55),
  'przygotowanie_cech__prog_korelacji': np.float64(0.7),
  'przygotowanie_cech__prog_monotonicznosci': np.float64(0.1),
  'przygotowanie_cech__prog_nan': np.float64(0.35),
  'przygotowanie_cech__prog_zer': np.float64(0.3),
  'przygotowanie_cech__tylko_wskazniki': np.False_},
 np.float64(0.7733512682068492))

In [63]:

najlepszy_model=model.best_estimator_
roc_auc_score(y_valid, najlepszy_model.predict_proba(X_valid)[:, 1])

0.8060975609756098

In [64]:
betas = najlepszy_model.named_steps["classifier"].coef_[0]
np.sum(betas > 0)

np.int64(0)

In [65]:
np.sum(betas != 0)

np.int64(15)